<a href="https://colab.research.google.com/github/ongsoony8382/financial-data-programming/blob/main/5%EC%A3%BC%EC%B0%A8_%EB%B3%B4%ED%97%98%ED%9A%8C%EC%82%AC_%EC%86%90%ED%95%B4%EC%9C%A8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 보험회사의 손해율 계산

# 용어 정의
- 보험료 (Premium): 보험계약자가 보험회사에 납입하는 금전 (보험회사의 수입)
- 보험금 (Claims 또는 Loss): 사고가 발생하여 보험회사가 피보험자에게 지급하는 금전(보험회사의 지출)

# 데이터 구조 및 특징 (Data Structure)
- 보험 데이터는 보통 두 개의 큰 테이블(Table)로 나뉘어 관리되며, 증권번호(Policy ID)를 통해 연결됨.

# 보험료 데이터 (Premium Data)
| 컬럼명 | 설명 | 예시 |
|---|---|---|
| Policy_ID | 증권번호 | 데이터값 예시 |
| Product | 보험상품 | 공장화재보험 |
| Premium | 연간보험료 | 1,250,000 |
| Start_Date | 보험시기 | 2026-03-01 |
| End_Date | 보험종기 | 2027-02-28 |
| ... | ... | ... |
...	...	...
# 보험금 데이터 (Claims Data, Loss Data)
| 컬럼명 | 설명 | 예시 |
|---|---|---|
| Claim_ID | 사고번호 | CLM-0025 |
| Policy_ID | 증권번호 | POL-2421 |
| Accident_Date | 사고날짜 | 2026-05-28 |
| Claim_Amount | 보험금 | 45,000,000 |
| Cause_of_Loss | 손해의 원인 | 공화구내폭발 |
| ... | ... | ... |

# 손해율 (Loss Ratio)
회사의 수익성을 판단하는 지표(%)

$$
\text{손해율(Loss Ratio)} = \frac{\text{보험금(Claims)}}{\text{보험료(Premiums)}} \times 100\%
$$


- 손해율 > 100%: 보험회사가 계약자로부터 받은 보험료보다 피보험자에게 지급한 보험금이 큰 경우 (보험회사의 손실)
- 손해율 < 100%: 보험회사가 계약자로부터 받은 보험료가 피보험자에게 지급한 보험금보다 큰 경우 (보험회사의 이익)


# (가상) 보험데이터 생성
- 보험회사의 데이터는 개인정보가 다수 포함되어 공개된 자료가 없으므로
보험료와 보험금 데이터를 임의로 생성하여 손해율을 계산

In [2]:
import pandas as pd
import numpy as np

# 1. 보험료(계약) 데이터 생성 (10명의 고객)
data_premium = { # 딕셔너리 안에 key 3개가 있고, 각 key의 값이 리스트인 구조
    'Policy_ID' : [f'POL_{i:03d}' for i in range(1,11)],
    'Customer_Name' : ['Kim', 'Lee', 'Park', 'Choi', 'Jung', 'Kang', 'Jo', 'Yoon', 'Jang', 'Lim'],
    'Annual_Premium' : [100, 120, 80, 200, 150, 90, 110, 130, 250, 100] # 단위: 만원
}

# 일렬로 표시된 딕셔너리를 표 형태로 보기 쉽게
df_premium = pd.DataFrame(data_premium)
print("[보험료 테이블]")
print(df_premium)

# 2. 보험금(사고) 데이터 생성 (일부 고객만 사고 발생)
data_claim = {
    'Policy_ID': ['POL_001', 'POL_004', 'POL_004', 'POL_009'], # 4번 고객은 사고 2번
    'Accident_Date' : ['2024-02-10', '2024-05-05', '2024-08-20', '2024-11-11'],
    'Claim_Amount': [50, 150, 300, 700] # 단위: 만원
}

df_claim = pd.DataFrame(data_claim)
print("\n[보험금 테이블]")
print(df_claim)

[보험료 테이블]
  Policy_ID Customer_Name  Annual_Premium
0   POL_001           Kim             100
1   POL_002           Lee             120
2   POL_003          Park              80
3   POL_004          Choi             200
4   POL_005          Jung             150
5   POL_006          Kang              90
6   POL_007            Jo             110
7   POL_008          Yoon             130
8   POL_009          Jang             250
9   POL_010           Lim             100

[보험금 테이블]
  Policy_ID Accident_Date  Claim_Amount
0   POL_001    2024-02-10            50
1   POL_004    2024-05-05           150
2   POL_004    2024-08-20           300
3   POL_009    2024-11-11           700


# 데이터 병합 (Merge / Left Join)
분석을 위해 두 테이블을 합쳐야 함.
pd.merge를 사용.

In [8]:
# Policy_ID를 기준으로 병합 (Left Join: 사고 안 난 사람도 포함)
df_merged = pd.merge(df_premium, df_claim, on='Policy_ID', how='left')

# 사고 안 난 사람(NaN)의 보험금은 0으로 채우기
df_merged['Claim_Amount'] = df_merged['Claim_Amount'].fillna(0)
# 사고 안 난 사람의 사고 일은 공백으로 채우기
df_merged['Accident_Date'].fillna(" ", inplace=True)

print("\n[보험료와 보험금의 병합된 데이터]")
print(df_merged)


[보험료와 보험금의 병합된 데이터]
   Policy_ID Customer_Name  Annual_Premium Accident_Date  Claim_Amount
0    POL_001           Kim             100    2024-02-10          50.0
1    POL_002           Lee             120                         0.0
2    POL_003          Park              80                         0.0
3    POL_004          Choi             200    2024-05-05         150.0
4    POL_004          Choi             200    2024-08-20         300.0
5    POL_005          Jung             150                         0.0
6    POL_006          Kang              90                         0.0
7    POL_007            Jo             110                         0.0
8    POL_008          Yoon             130                         0.0
9    POL_009          Jang             250    2024-11-11         700.0
10   POL_010           Lim             100                         0.0


/tmp/ipykernel_2104/1381126526.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_merged['Accident_Date'].fillna(" ", inplace=True)


# 손해율 분석
개별 보험계약자가 아닌, 공장화재보험 전체 포트폴리오의 손해율을 계산

In [9]:
total_premium = df_merged['Annual_Premium'].sum() # 총 수입
total_claim = df_merged['Claim_Amount'].sum() # 총 지출

loss_ratio = (total_claim / total_premium) * 100

print(f"=== 포트폴리오 분석 결과 ===")
print(f"총 수입 보험료: {total_premium} 만원")
print(f"총 지급 보험금: {total_claim} 만원")
print(f"손해율(Loss Ratio): {loss_ratio:.2f}%")

if loss_ratio > 100:
  print("상태: 적자 (보험료 인상 필요)")
else :
  print("상태: 흑자 (안정적 운영 중)")

=== 포트폴리오 분석 결과 ===
총 수입 보험료: 1530 만원
총 지급 보험금: 1200.0 만원
손해율(Loss Ratio): 78.43%
상태: 흑자 (안정적 운영 중)
